In [43]:
from pathlib import Path
import numpy as np
import pandas as pd
print("pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

pandas version: 2.1.4
NumPy version: 1.26.4


In [44]:
PROJECT_ROOT = Path(
"../"
)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Processed-data folder:", PROCESSED_DIR)
print("Folder exists:", PROCESSED_DIR.exists())

Project root: ..
Processed-data folder: ../data/processed
Folder exists: True


In [45]:
INPUT_FILE = PROCESSED_DIR / "student-mat-clean.csv"
print("Input file:", INPUT_FILE)
print("File exists:", INPUT_FILE.exists())

Input file: ../data/processed/student-mat-clean.csv
File exists: True


In [46]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Could not find the input file:\n{INPUT_FILE}\n"
        "Check PROJECT_ROOT and the input filename."
    )

df = pd.read_csv(
    INPUT_FILE,
    sep=None,
    engine="python"
)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
display(df.head())

Dataset loaded successfully.
Dataset shape: (395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [47]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

Number of rows: 395
Number of columns: 33

Column names:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


In [48]:
dtype_table = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values
    }
)

display(dtype_table)

,column,dtype
0,school,object
1,sex,object
2,age,int64
3,address,object
4,famsize,object
5,Pstatus,object
6,Medu,int64
7,Fedu,int64
8,Mjob,object
9,Fjob,object


In [49]:
missing_summary = (
    df.isna()
        .sum()
        .sort_values(ascending=False)
        .to_frame("missing_count")
)

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(2)

display(missing_summary.head(20))

,missing_count,missing_percent
school,0,0.0
paid,0,0.0
G2,0,0.0
G1,0,0.0
absences,0,0.0
health,0,0.0
Walc,0,0.0
Dalc,0,0.0
goout,0,0.0
freetime,0,0.0


In [50]:
print("Total missing values before encoding:", df.isna().sum().sum())

Total missing values before encoding: 0


In [51]:
cat_cols = df.select_dtypes(include="object").columns
print("Categorical columns:")
print(list(cat_cols))
print("\nNumber of categorical columns:", len(cat_cols))

Categorical columns:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

Number of categorical columns: 17


In [52]:
if len(cat_cols) == 0:
    raise ValueError(
        "No object-type categorical columns were found. "
        "The dataset may already be encoded, or the categorical "
        "columns may use a different data type."
    )

print("Categorical columns were found. Encoding can continue.")

Categorical columns were found. Encoding can continue.


In [53]:
for column in cat_cols:
    unique_values = df[column].drop_duplicates().tolist()

    print("=" * 70)
    print("Column:", column)
    print("Number of nonmissing categories:", df[column].nunique(dropna=True))
    print("Number of missing values:", df[column].isna().sum())
    print("Example categories:", unique_values[:10])

Column: school
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['GP', 'MS']
Column: sex
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['F', 'M']
Column: address
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['U', 'R']
Column: famsize
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['GT3', 'LE3']
Column: Pstatus
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['A', 'T']
Column: Mjob
Number of nonmissing categories: 5
Number of missing values: 0
Example categories: ['at_home', 'health', 'other', 'services', 'teacher']
Column: Fjob
Number of nonmissing categories: 5
Number of missing values: 0
Example categories: ['teacher', 'other', 'services', 'health', 'at_home']
Column: reason
Number of nonmissing categories: 4
Number of missing values: 0
Example categories: ['course', 'other', 'home', 'reputation']
Column: g

In [54]:
possible_identifier_columns = []

for column in cat_cols:
    uniqueness_ratio = df[column].nunique(dropna=False) / len(df)

    if uniqueness_ratio > 0.90:
        possible_identifier_columns.append(
            {
                "column": column,
                "unique_values": df[column].nunique(dropna=False),
                "uniqueness_ratio": round(uniqueness_ratio, 3)
            }
        )

if possible_identifier_columns:
    print("Review these possible identifier columns before encoding:")
    display(pd.DataFrame(possible_identifier_columns))
else:
    print("No likely identifier columns were detected.")

No likely identifier columns were detected.


In [55]:
rows_before = df.shape[0]
columns_before = df.shape[1]

print("Rows before encoding:", rows_before)
print("Columns before encoding:", columns_before)

Rows before encoding: 395
Columns before encoding: 33


In [56]:
category_count_table = []

for column in cat_cols:
    category_count = df[column].nunique(dropna=True)
    dummy_count = max(category_count - 1, 0)

    category_count_table.append(
        {
            "original_column": column,
            "number_of_categories": category_count,
            "expected_dummy_columns": dummy_count
        }
    )

category_count_df = pd.DataFrame(category_count_table)

display(category_count_df)

,original_column,number_of_categories,expected_dummy_columns
0,school,2,1
1,sex,2,1
2,address,2,1
3,famsize,2,1
4,Pstatus,2,1
5,Mjob,5,4
6,Fjob,5,4
7,reason,4,3
8,guardian,3,2
9,schoolsup,2,1


In [57]:
number_of_numeric_columns = df.shape[1] - len(cat_cols)

expected_dummy_columns = (
    category_count_df["expected_dummy_columns"].sum()
)

expected_total_columns = (
    number_of_numeric_columns + expected_dummy_columns
)

print("Original numeric columns:", number_of_numeric_columns)
print("Expected dummy columns:", expected_dummy_columns)
print("Expected final column count:", expected_total_columns)

Original numeric columns: 16
Expected dummy columns: 26
Expected final column count: 42


In [58]:
cat_cols = df.select_dtypes(include="object").columns

df_encoded = pd.get_dummies(
    df,
    columns=list(cat_cols),
    drop_first=True
)

print("Encoded shape:", df_encoded.shape)

Encoded shape: (395, 42)


In [59]:
rows_after = df_encoded.shape[0]
columns_after = df_encoded.shape[1]

print("Rows before encoding:", rows_before)
print("Rows after encoding:", rows_after)

print("\nColumns before encoding:", columns_before)
print("Columns after encoding:", columns_after)

print("\nNet change in columns:", columns_after - columns_before)

Rows before encoding: 395
Rows after encoding: 395

Columns before encoding: 33
Columns after encoding: 42

Net change in columns: 9


In [60]:
print("Expected final column count:", expected_total_columns)
print("Actual final column count:", columns_after)

if columns_after == expected_total_columns:
    print("Column-count check passed.")
else:
    print(
        "Column-count check requires review. "
        "Missing values or unusual category structures may affect the result."
    )

Expected final column count: 42
Actual final column count: 42
Column-count check passed.


In [61]:
original_columns = set(df.columns)
encoded_columns = set(df_encoded.columns)

removed_columns = [
    column for column in df.columns
    if column not in encoded_columns
]

new_columns = [
    column for column in df_encoded.columns
    if column not in original_columns
]

print("Original categorical columns removed:")
print(removed_columns)

print("\nNumber of new dummy columns:", len(new_columns))
print("\nFirst 30 new dummy columns:")
print(new_columns[:30])

Original categorical columns removed:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

Number of new dummy columns: 26

First 30 new dummy columns:
['school_MS', 'sex_M', 'address_U', 'famsize_LE3', 'Pstatus_T', 'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other', 'schoolsup_yes', 'famsup_yes', 'paid_yes', 'activities_yes', 'nursery_yes', 'higher_yes', 'internet_yes', 'romantic_yes']


In [62]:
display(df_encoded.head())

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,True,False,True,False,False,False,True,True,False,False
1,17,1,1,1,2,0,5,3,3,1,...,False,False,False,True,False,False,False,True,True,False
2,15,1,1,1,2,3,4,3,2,2,...,True,False,True,False,True,False,True,True,True,False
3,15,4,2,1,3,0,3,2,2,1,...,True,False,False,True,True,True,True,True,True,True
4,16,3,3,1,2,0,4,3,2,1,...,False,False,False,True,True,False,True,True,False,False


In [63]:
display(df_encoded[new_columns].head())

,school_MS,sex_M,address_U,famsize_LE3,Pstatus_T,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,False,False,True,False,False,False,False,False,False,False,...,True,False,True,False,False,False,True,True,False,False
1,False,False,True,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,True,False
2,False,False,True,True,True,False,False,False,False,False,...,True,False,True,False,True,False,True,True,True,False
3,False,False,True,False,True,True,False,False,False,False,...,True,False,False,True,True,True,True,True,True,True
4,False,False,True,False,True,False,True,False,False,False,...,False,False,False,True,True,False,True,True,False,False


In [64]:
boolean_columns = df_encoded.select_dtypes(include="bool").columns
print("Number of Boolean dummy columns:", len(boolean_columns))

Number of Boolean dummy columns: 26


In [65]:
df_encoded[boolean_columns] = (
    df_encoded[boolean_columns]
    .astype("int8")
)

print("Boolean dummy variables converted to 0 and 1.")

Boolean dummy variables converted to 0 and 1.


In [66]:
display(df_encoded.head())

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [67]:
remaining_object_columns = (
    df_encoded
    .select_dtypes(include="object")
    .columns
    .tolist()
)

print("Remaining object columns:", remaining_object_columns)

Remaining object columns: []


In [68]:
assert len(remaining_object_columns) == 0, (
    "Some object-type columns remain after encoding."
)

print("Object-column check passed.")

Object-column check passed.


In [69]:
assert df_encoded.shape[0] == df.shape[0], (
    "The number of rows changed during encoding."
)

print("Row-count check passed.")

Row-count check passed.


In [70]:
assert df_encoded.index.equals(df.index), (
    "The row index changed during encoding."
)

print("Row-index check passed.")

Row-index check passed.


In [71]:
if "G3" in df.columns:
    assert "G3" in df_encoded.columns, (
        "The G3 target variable is missing after encoding."
    )

    pd.testing.assert_series_equal(
        df_encoded["G3"],
        df["G3"],
        check_names=True
    )

    print("G3 target-variable check passed.")
else:
    print("G3 was not found in the current dataset.")

G3 target-variable check passed.


In [72]:
duplicate_column_count = (
    df_encoded.columns.duplicated().sum()
)

print("Duplicate column names:", duplicate_column_count)

assert duplicate_column_count == 0, (
    "Duplicate column names were detected."
)

print("Duplicate-column check passed.")

Duplicate column names: 0
Duplicate-column check passed.


In [73]:
missing_after = df_encoded.isna().sum().sum()
print("Total missing values before encoding:", df.isna().sum().sum())
print("Total missing values after encoding:", missing_after)

Total missing values before encoding: 0
Total missing values after encoding: 0


In [74]:
columns_with_missing_values = (
    df_encoded.isna()
    .sum()
    .loc[lambda values: values > 0]
    .sort_values(ascending=False)
)

if columns_with_missing_values.empty:
    print("No missing values remain.")
else:
    print("Columns still containing missing values:")
    display(columns_with_missing_values.to_frame("missing_count"))

No missing values remain.


In [75]:
assert df_encoded.shape[0] == df.shape[0], (
    "Sanity check failed: row count changed."
)
assert df_encoded.index.equals(df.index), (
    "Sanity check failed: row index changed."
)

assert not df_encoded.columns.duplicated().any(), (
    "Sanity check failed: duplicate column names exist."
)

remaining_objects = (
    df_encoded.select_dtypes(include="object").columns.tolist()
)

assert len(remaining_objects) == 0, (
    f"Sanity check failed: object columns remain: {remaining_objects}"
)

if "G3" in df.columns:
    assert "G3" in df_encoded.columns, (
        "Sanity check failed: G3 is missing."
    )

    assert df_encoded["G3"].equals(df["G3"]), (
        "Sanity check failed: G3 values changed."
    )

print("All one-hot encoding sanity checks passed.")

All one-hot encoding sanity checks passed.


In [76]:
encoding_summary = pd.DataFrame(
    {
        "measurement": [
            "Rows before encoding",
            "Rows after encoding",
            "Columns before encoding",
            "Columns after encoding",
            "Categorical columns encoded",
            "New dummy columns",
            "Remaining object columns",
            "Missing values after encoding"
        ],
        "value": [
            df.shape[0],
            df_encoded.shape[0],
            df.shape[1],
            df_encoded.shape[1],
            len(cat_cols),
            len(new_columns),
            len(remaining_object_columns),
            df_encoded.isna().sum().sum()
        ]
    }
)

display(encoding_summary)

,measurement,value
0,Rows before encoding,395
1,Rows after encoding,395
2,Columns before encoding,33
3,Columns after encoding,42
4,Categorical columns encoded,17
5,New dummy columns,26
6,Remaining object columns,0
7,Missing values after encoding,0


In [77]:
OUTPUT_FILE = PROCESSED_DIR / "student-mat-encoded.csv"

df_encoded.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Encoded dataset saved successfully.")
print("Output file:", OUTPUT_FILE)

Encoded dataset saved successfully.
Output file: ../data/processed/student-mat-encoded.csv


In [78]:
assert OUTPUT_FILE.exists(), (
    f"The encoded output file was not created: {OUTPUT_FILE}"
)

saved_size = OUTPUT_FILE.stat().st_size

print("Saved file exists:", OUTPUT_FILE.exists())
print("Saved file size:", saved_size, "bytes")

Saved file exists: True
Saved file size: 34846 bytes


In [79]:
df_encoded_check = pd.read_csv(OUTPUT_FILE)

print("Original encoded shape:", df_encoded.shape)
print("Reloaded encoded shape:", df_encoded_check.shape)

assert df_encoded_check.shape == df_encoded.shape, (
    "The reloaded dataset shape does not match."
)

print("Saved-file verification passed.")

Original encoded shape: (395, 42)
Reloaded encoded shape: (395, 42)
Saved-file verification passed.


# One-Hot Encoding Review

## Summary

Your one-hot encoding step appears to have worked correctly based on the information provided.

- **Original dataset shape:** `(395, 33)`
- **Encoded dataset shape:** `(395, 42)`
- **Rows after encoding:** **395** (unchanged, which is expected)
- **Object columns remaining:** **None** (all categorical variables were encoded)
- **New dummy columns created:** **26**

The code used is a standard and appropriate way to one-hot encode categorical variables for many machine learning models.

```python
cat_cols = df.select_dtypes(include="object").columns

df_encoded = pd.get_dummies(
    df,
    columns=list(cat_cols),
    drop_first=True
)

print("Encoded shape:", df_encoded.shape)
```

---

## Interpretation

### 1. Did the categorical columns expand correctly?

Yes.

The original dataset contained **17 categorical (`object`) columns**:

- `school`
- `sex`
- `address`
- `famsize`
- `Pstatus`
- `Mjob`
- `Fjob`
- `reason`
- `guardian`
- `schoolsup`
- `famsup`
- `paid`
- `activities`
- `nursery`
- `higher`
- `internet`
- `romantic`

After running `pd.get_dummies()`:

- All object columns were replaced with numeric dummy variables.
- No object columns remain.
- New binary (0/1) columns were created.

This is the expected result of one-hot encoding.

---

### 2. Why did the number of columns increase?

Each categorical variable is converted into one or more binary (0/1) columns.

For example, the original column:

| school |
|--------|
| GP |
| MS |

becomes:

| school_MS |
|-----------|
| 0 |
| 1 |

Variables with more than two categories produce multiple dummy columns.

For example:

| reason |
|---------|
| home |
| course |
| reputation |
| other |

becomes:

| reason_course | reason_home | reason_reputation |
|----------------|-------------|-------------------|
| 0/1 | 0/1 | 0/1 |

The omitted category (`other`) is represented whenever all three dummy variables equal **0**.

Because one original column can become several new columns, the total number of columns increases.

---

### 3. What is the purpose of `drop_first=True`?

The parameter `drop_first=True` removes one dummy column for each categorical variable.

Without `drop_first=True`:

- A variable with **k** categories creates **k** dummy columns.

With `drop_first=True`:

- A variable with **k** categories creates **k − 1** dummy columns.

This prevents **perfect multicollinearity**, also known as the **dummy variable trap**, where one dummy variable can be perfectly predicted from the others.

This is especially important for models such as:

- Linear Regression
- Logistic Regression

Although tree-based models (Decision Trees, Random Forests, Gradient Boosting) are less affected, using `drop_first=True` is still common practice.

---

### 4. How does the omitted category become the reference category?

The category that is dropped becomes the **reference (baseline) category**.

For example, suppose the `guardian` column contains:

- father
- mother
- other

After encoding with `drop_first=True`, the dataset may contain:

- `guardian_mother`
- `guardian_other`

If a student's encoded values are:

| guardian_mother | guardian_other |
|-----------------|----------------|
| 0 | 0 |

then the student's guardian is:

```
father
```

The remaining dummy variables are interpreted relative to this baseline.

For example:

- `guardian_mother = 1` means the guardian is **mother** instead of **father**.
- `guardian_other = 1` means the guardian is **other** instead of **father**.

The omitted category is therefore still represented—it is encoded implicitly.

---

### 5. Suggested sanity checks

After encoding, perform a few quick checks to confirm everything worked correctly.

#### Check for remaining object columns

```python
print(df_encoded.select_dtypes(include="object").columns)
```

Expected output:

```python
Index([], dtype='object')
```

---

#### Verify the dataset shape

```python
print(df_encoded.shape)
```

The output should confirm that the dataset still contains **395 rows**.

---

#### Inspect the new column names

```python
print(df_encoded.columns)
```

You should see new columns similar to:

- `school_MS`
- `sex_M`
- `address_U`
- `guardian_mother`
- `reason_reputation`

---

#### Check for missing values

```python
print(df_encoded.isnull().sum().sum())
```

Ideally, the result should be:

```
0
```

unless the original dataset already contained missing values.

---

### 6. Should the number of rows remain unchanged?

Yes.

One-hot encoding changes the **columns**, not the **rows**.

Each student remains one observation, so the dataset should still contain:

- **395 rows before encoding**
- **395 rows after encoding**

If the number of rows changes, another preprocessing step—not one-hot encoding—is responsible.

---

### 7. Possible risks

Although the encoding appears correct, there are several potential issues to consider.

#### Missing categories in future data

New data may contain categories that were not present during training.

For example:

Training data:

```
reason
```

contains:

- home
- course
- reputation

Later, new data contains:

```
reason = internship
```

The model will not know how to encode this unseen category unless the preprocessing pipeline is designed to handle unknown values.

---

#### Identifier columns

Identifier columns (such as Student ID or Record Number) should generally **not** be one-hot encoded because they do not provide useful predictive information.

The Student Performance dataset does not contain identifier columns, so this is not currently an issue.

---

#### High-cardinality variables

Variables with many unique categories (such as hundreds of cities or names) can generate hundreds of dummy variables.

This can:

- increase memory usage,
- slow model training,
- increase the risk of overfitting.

The Student Performance dataset contains only a small number of categories per variable, so this is not a concern.

---

#### Inconsistent category labels

One-hot encoding treats different spellings as different categories.

For example:

- `GP`
- `gp`
- ` GP`

would all become separate dummy variables.

Before encoding, categorical data should be cleaned by removing extra spaces and ensuring consistent capitalization.

---

## Recommendation

The one-hot encoding process appears to have been completed successfully.

Key observations:

- ✅ All categorical variables were converted into numeric dummy variables.
- ✅ The number of rows remained unchanged (**395**), which is expected.
- ✅ The increase from **33** to **42** columns is normal because categorical variables expand into multiple binary columns.
- ✅ Using `drop_first=True` avoids redundant dummy variables and establishes a reference category for each encoded feature.
- ✅ No object-type columns remain, indicating that the dataset is ready for most machine learning algorithms.

Before training your models, it is recommended to:

1. Verify that there are no missing values after preprocessing.
2. Review the new dummy variable names to ensure they match the expected categories.
3. Apply the same encoding process to future or test datasets so that the feature columns remain consistent.
4. Be prepared to handle unseen categories in future data by using the same fitted preprocessing pipeline during model deployment.